# How do tree-based models make decisions?
**Topics:** Decision Trees · Random Forests · Bagging · Boosting · XGBoost · LightGBM · CatBoost


---
## 1. Decision Trees

### What it is
- Splits data recursively on feature thresholds to create a tree of decisions
- Each leaf node gives a prediction (class label or mean value)
- Works for both classification and regression

### Key intuition
- At each node, find the split that best separates the data
- "Best" = minimizes impurity (Gini / entropy for classification, MSE for regression)
- Keeps splitting until stopping criteria met (max depth, min samples, etc.)

### Splitting criteria

| Criterion | Task | Formula (simplified) |
|---|---|---|
| Gini impurity | Classification | 1 - Σ(pᵢ²) |
| Entropy | Classification | -Σ(pᵢ log pᵢ) |
| MSE | Regression | mean((y - ȳ)²) |

### When to use
- Need an interpretable model you can visualize
- Mixed feature types (numerical + categorical)
- Baseline before trying ensemble methods

### When not to use
- Need high accuracy — single trees overfit easily
- Smooth decision boundaries needed — trees create rectangular regions


In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=500, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── Basic tree ──
tree = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
tree.fit(X_train, y_train)

print(f"Train accuracy: {accuracy_score(y_train, tree.predict(X_train)):.3f}")
print(f"Test accuracy : {accuracy_score(y_test,  tree.predict(X_test)):.3f}")
print(f"Tree depth    : {tree.get_depth()}")
print(f"Leaf nodes    : {tree.get_n_leaves()}")

# Feature importance
importances = tree.feature_importances_
top = np.argsort(importances)[::-1][:5]
print(f"\nTop 5 features: {top}")
print(f"Importances   : {importances[top].round(3)}")

# Visualize tree rules (text)
rules = export_text(tree, max_depth=2)
print(f"\nTree structure (depth 2):\n{rules}")

# Effect of max_depth on overfitting
print(f"{'Depth':<8} {'Train Acc':<12} {'Test Acc'}")
for d in [1, 2, 3, 5, 10, None]:
    t = DecisionTreeClassifier(max_depth=d, random_state=42)
    t.fit(X_train, y_train)
    print(f"{str(d):<8} {accuracy_score(y_train, t.predict(X_train)):<12.3f} {accuracy_score(y_test, t.predict(X_test)):.3f}")


### Interview Q&A

**Gini vs entropy — which to use?**
- Both work similarly in practice — Gini is slightly faster to compute
- Entropy can produce slightly more balanced trees

**How does a decision tree handle overfitting?**
- Limit max_depth, min_samples_split, min_samples_leaf
- Post-pruning: grow full tree, then prune branches that don't improve validation performance

**Why do decision trees create rectangular decision boundaries?**
- Each split is on a single feature threshold → axis-aligned partitions
- Can approximate any boundary but needs many splits for curved ones

### Gotchas
- Unstable — small changes in data can produce very different trees
- Biased toward features with more unique values (high cardinality)
- Always tune max_depth — unconstrained trees always overfit


---
## 2. Bagging & Random Forests

### What it is
- **Bagging** → train many models on random subsets of data (bootstrap samples), average predictions
- **Random Forest** → bagging + random feature subsets at each split → reduces correlation between trees
- Ensemble of decision trees that vote (classification) or average (regression)

### Key intuition
- One tree is unstable and overfits. Many diverse trees average out each other's errors
- Random feature subsets force trees to be different → diversity → better generalization
- Out-of-bag (OOB) samples (not used in bootstrap) give free validation estimate

### When to use
- Tabular data — one of the best out-of-the-box algorithms
- Need feature importances
- Don't want to tune many hyperparameters

### When not to use
- Need highly interpretable model — forest of 500 trees is a black box
- Very high-dimensional sparse data (text) — gradient boosting or linear models often better
- Real-time inference with tight latency — many trees = slow prediction


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── Bagging (manual) ──
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.8,      # 80% of data per tree
    max_features=0.8,     # 80% of features per tree
    random_state=42
)
bagging.fit(X_train, y_train)
print(f"Bagging test accuracy   : {accuracy_score(y_test, bagging.predict(X_test)):.3f}")

# ── Random Forest ──
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,        # trees grow fully
    max_features='sqrt',   # sqrt(n_features) per split — default for classification
    min_samples_leaf=1,
    oob_score=True,        # free validation using out-of-bag samples
    random_state=42
)
rf.fit(X_train, y_train)
print(f"Random Forest test acc  : {accuracy_score(y_test, rf.predict(X_test)):.3f}")
print(f"OOB score (free val)    : {rf.oob_score_:.3f}")

# Feature importances
importances = rf.feature_importances_
top5 = np.argsort(importances)[::-1][:5]
print(f"\nTop 5 feature importances:")
for i in top5:
    print(f"  Feature {i:2d}: {importances[i]:.4f}")


### Interview Q&A

**Why does Random Forest outperform a single Decision Tree?**
- Averaging many uncorrelated trees reduces variance without increasing bias
- Random feature subsets decorrelate trees — they make different errors

**What is out-of-bag score?**
- Each tree is trained on ~63% of data (bootstrap sample)
- Remaining ~37% (OOB) are used to evaluate that tree
- Aggregate OOB predictions = free validation estimate, no separate val set needed

**max_features in Random Forest — what to set?**
- Classification: sqrt(n_features) — default
- Regression: n_features / 3 — default
- Lower = more diverse trees, higher variance reduction but slower

### Gotchas
- More trees = better up to a point, then diminishing returns (usually 100-500 is enough)
- Feature importances can be biased toward high-cardinality features — use permutation importance instead
- Random Forest doesn't extrapolate — predictions are bounded by training target range


---
## 3. Boosting

### What it is
- Ensemble method that trains models **sequentially** — each new model corrects errors of the previous
- Final prediction = weighted sum of all models
- Reduces bias (unlike bagging which reduces variance)

### Key intuition
- Each tree focuses on the mistakes the previous trees made
- Misclassified points get higher weight → next tree pays more attention to them
- AdaBoost → reweights samples. Gradient Boosting → fits residuals (errors)

### AdaBoost vs Gradient Boosting

| | AdaBoost | Gradient Boosting |
|---|---|---|
| Corrects by | Reweighting samples | Fitting residuals |
| Base learner | Usually shallow trees (stumps) | Shallow trees |
| Robust to outliers | No — outliers get high weight | More robust |
| Modern use | Less common | Foundation for XGBoost/LGB |

### When to use
- Tabular data where you need high accuracy
- You have time to tune hyperparameters

### When not to use
- Noisy data with many outliers — boosting can overfit noise
- Need fast training — sequential nature limits parallelism


In [ ]:
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── AdaBoost ──
ada = AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
ada.fit(X_train, y_train)
print(f"AdaBoost accuracy         : {accuracy_score(y_test, ada.predict(X_test)):.3f}")

# ── Gradient Boosting ──
gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,           # shallow trees are key
    subsample=0.8,         # stochastic gradient boosting
    random_state=42
)
gb.fit(X_train, y_train)
print(f"Gradient Boosting accuracy: {accuracy_score(y_test, gb.predict(X_test)):.3f}")

# Staged predictions — see how accuracy improves with more trees
staged_scores = [accuracy_score(y_test, pred) for pred in gb.staged_predict(X_test)]
print(f"\nAccuracy after 10 trees : {staged_scores[9]:.3f}")
print(f"Accuracy after 50 trees : {staged_scores[49]:.3f}")
print(f"Accuracy after 100 trees: {staged_scores[99]:.3f}")


### Interview Q&A

**Bagging vs Boosting?**
- Bagging → parallel, reduces variance, good for high-variance models (deep trees)
- Boosting → sequential, reduces bias, good for high-bias models (shallow trees)

**What is the learning rate in boosting?**
- Scales the contribution of each tree
- Lower learning rate → need more trees → slower but often better generalization
- Tune learning rate and n_estimators together

**Can boosting overfit?**
- Yes — especially with deep trees or too many estimators on noisy data
- Use early stopping, lower learning rate, limit max_depth

### Gotchas
- Gradient boosting is slow to train on large data — use XGBoost or LightGBM instead
- Learning rate and n_estimators are tightly coupled — halve LR, double n_estimators
- subsample < 1.0 (stochastic GB) often improves generalization and speed


---
## 4. XGBoost

### What it is
- Optimized gradient boosting library — faster, more regularized, more scalable
- Adds L1/L2 regularization on tree weights (unlike vanilla gradient boosting)
- Supports parallel tree construction, GPU training, missing value handling

### Key intuition
- Same idea as gradient boosting (fit residuals sequentially)
- Key improvements: second-order gradients for better optimization, built-in regularization, efficient histogram-based splitting

### When to use
- Structured/tabular data competition — often the winning algorithm
- Medium to large datasets
- Need built-in regularization to control overfitting

### When not to use
- Very large datasets with millions of rows → LightGBM is faster
- High-cardinality categorical features → CatBoost handles them better natively
- Unstructured data (images, text) → deep learning


In [ ]:
# pip install xgboost
try:
    import xgboost as xgb
    from sklearn.datasets import make_classification
    from sklearn.model_selection import train_test_split, cross_val_score
    from sklearn.metrics import accuracy_score
    import numpy as np

    X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # ── Basic XGBoost ──
    model = xgb.XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,   # fraction of features per tree
        reg_alpha=0.1,          # L1 regularization
        reg_lambda=1.0,         # L2 regularization
        eval_metric='logloss',
        random_state=42,
        verbosity=0
    )

    # Early stopping to find optimal n_estimators
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    print(f"XGBoost test accuracy: {accuracy_score(y_test, model.predict(X_test)):.3f}")
    print(f"Best iteration       : {model.best_iteration}")

    # Cross-validation
    cv_scores = cross_val_score(
        xgb.XGBClassifier(n_estimators=100, verbosity=0, random_state=42),
        X, y, cv=5, scoring='accuracy'
    )
    print(f"CV accuracy: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")

    # Feature importance
    importances = model.feature_importances_
    top5 = np.argsort(importances)[::-1][:5]
    print(f"\nTop 5 features: {top5}")

except ImportError:
    print("Install xgboost: pip install xgboost")
    print("Key params: n_estimators, learning_rate, max_depth, subsample, colsample_bytree, reg_alpha, reg_lambda")


### Interview Q&A

**XGBoost vs sklearn GradientBoosting?**
- XGBoost is faster (parallelized), has built-in regularization, handles missing values natively
- sklearn GB is simpler but slower and less configurable

**What is colsample_bytree?**
- Fraction of features randomly sampled per tree (like max_features in RF)
- Adds randomness → reduces overfitting, similar to Random Forest's feature sampling

**How do you tune XGBoost?**
1. Fix learning_rate=0.1, tune max_depth (3-6) and min_child_weight
2. Tune subsample and colsample_bytree (0.6-0.9)
3. Add regularization (reg_alpha, reg_lambda)
4. Lower learning_rate and increase n_estimators with early stopping

### Gotchas
- Always use early stopping — prevents overfitting without manual tuning of n_estimators
- scale_pos_weight for imbalanced classification: set to (negative samples / positive samples)
- XGBoost can handle missing values — don't impute blindly before using it


---
## 5. LightGBM

### What it is
- Gradient boosting framework by Microsoft — optimized for speed and memory on large datasets
- Uses **leaf-wise** tree growth (vs level-wise in XGBoost) — grows the most impactful leaf first
- Key innovations: Gradient-based One-Side Sampling (GOSS), Exclusive Feature Bundling (EFB)

### Key intuition
- Level-wise (XGBoost): grow all leaves at same depth — safe but slow
- Leaf-wise (LightGBM): grow the leaf with the biggest loss reduction — faster, can overfit on small data
- Histogram-based splitting: bucket continuous features into bins → much faster

### When to use
- Large datasets (100k+ rows) where XGBoost is too slow
- Memory-constrained environments
- High-cardinality categorical features (native support)

### When not to use
- Small datasets (< 10k rows) → leaf-wise growth can overfit, XGBoost safer
- Need very interpretable model


In [ ]:
# pip install lightgbm
try:
    import lightgbm as lgb
    from sklearn.datasets import make_classification
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score

    X, y = make_classification(n_samples=5000, n_features=30, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # ── LightGBM sklearn API ──
    model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=-1,           # no limit — controlled by num_leaves
        num_leaves=31,          # key param: 2^max_depth for balanced tree
        min_child_samples=20,   # min samples in leaf — prevents overfitting
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        verbose=-1,
        random_state=42
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )

    print(f"LightGBM test accuracy : {accuracy_score(y_test, model.predict(X_test)):.3f}")
    print(f"Best iteration         : {model.best_iteration_}")
    print(f"n_features importance  : {len(model.feature_importances_)}")

except ImportError:
    print("Install lightgbm: pip install lightgbm")
    print("Key params: num_leaves, min_child_samples, learning_rate, n_estimators")


### Interview Q&A

**LightGBM vs XGBoost — which to choose?**
- Large data (>100k rows) → LightGBM (faster, less memory)
- Small/medium data → XGBoost (more stable, less prone to overfit)
- Categorical features → LightGBM or CatBoost (native handling)

**What is num_leaves and why does it matter?**
- Controls model complexity — key hyperparameter in LightGBM
- More leaves → more complex model → risk of overfitting
- Rule of thumb: num_leaves < 2^max_depth

**What is GOSS?**
- Gradient-based One-Side Sampling — keeps all high-gradient samples, randomly samples low-gradient ones
- Reduces data size while preserving informative samples → faster without much accuracy loss

### Gotchas
- Leaf-wise growth can severely overfit small datasets — increase min_child_samples
- num_leaves is more important than max_depth in LightGBM
- Use early stopping always — LightGBM trains very fast so it's easy to overtrain


---
## 6. CatBoost

### What it is
- Gradient boosting library by Yandex — designed specifically for categorical features
- Uses **ordered boosting** to prevent target leakage during training
- Symmetric (oblivious) trees — same split condition at every node of a level

### Key intuition
- Traditional boosting with categorical features: you must one-hot encode or label encode → loses information or introduces leakage
- CatBoost uses **target statistics** (mean of target per category) computed in an order-preserving way → no leakage, better encoding
- Oblivious trees: fast to build, less prone to overfitting, efficient at inference

### When to use
- Dataset has many categorical features
- Want minimal preprocessing — CatBoost handles categoricals natively
- Need fast inference (oblivious trees are very cache-friendly)

### When not to use
- All features are numerical → XGBoost or LightGBM are often faster
- Very large datasets → LightGBM is more memory efficient


In [ ]:
# pip install catboost
try:
    from catboost import CatBoostClassifier
    from sklearn.datasets import make_classification
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score
    import numpy as np
    import pandas as pd

    # Simulate dataset with categorical features
    np.random.seed(42)
    n = 1000
    df = pd.DataFrame({
        'num1'     : np.random.randn(n),
        'num2'     : np.random.randn(n),
        'category1': np.random.choice(['A', 'B', 'C', 'D'], n),
        'category2': np.random.choice(['X', 'Y', 'Z'], n),
    })
    y = (df['num1'] + df['num2'] > 0).astype(int).values
    X = df

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    cat_features = ['category1', 'category2']  # tell CatBoost which columns are categorical

    model = CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=4,
        cat_features=cat_features,
        eval_metric='Accuracy',
        random_seed=42,
        verbose=0
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_test, y_test),
        early_stopping_rounds=50
    )

    print(f"CatBoost test accuracy: {accuracy_score(y_test, model.predict(X_test)):.3f}")
    print(f"Best iteration        : {model.best_iteration_}")

except ImportError:
    print("Install catboost: pip install catboost")
    print("Key advantage: pass cat_features list — no encoding needed")
    print("Key params: iterations, learning_rate, depth, cat_features")


### Interview Q&A

**CatBoost vs LightGBM for categorical features?**
- CatBoost → handles high-cardinality categoricals natively with no leakage, less preprocessing
- LightGBM → also supports categoricals but requires integer encoding first

**What is ordered boosting?**
- Standard target encoding leaks future information — uses all samples including the current one
- CatBoost processes samples in a random order and computes target statistics only from preceding samples → eliminates leakage

**When would you choose CatBoost over XGBoost?**
- Dataset has many categorical columns with high cardinality
- You want to minimize preprocessing effort
- Need fast and consistent inference (oblivious trees)

### Gotchas
- CatBoost trains slower than LightGBM — use GPU if available (`task_type='GPU'`)
- Must explicitly specify cat_features — CatBoost won't auto-detect them from dtype
- Default depth=6 can overfit small datasets — reduce to 4
